**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# RLS & Recursive Estimation

The rung between [APA](./Intro_AdFilt_APA.ipynb) and the [Kalman filter](./Intro_AdFilt_KF.ipynb): Recursive Least Squares solves the *entire* least-squares problem at every sample — exactly, recursively, without ever re-inverting a matrix — and turns out to be a Kalman filter wearing a different hat.

## 1. Pre-requisites

- [Adaptive Filtering: APA](./Intro_AdFilt_APA.ipynb) (the setup and its notation).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2 (normal equations).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# same system-identification scenario as the APA workshop
M = 16
w_true = np.exp(-0.4*np.arange(M)) * np.cos(0.9*np.arange(M)); w_true /= np.linalg.norm(w_true)
N = 3000
from scipy import signal as sig
x = sig.lfilter([1.0], [1.0, -0.9], rng.standard_normal(N))     # correlated input (the hard case)
d = np.convolve(x, w_true)[:N] + 0.01*rng.standard_normal(N)

---
### 🕐 Session 1 of 2 — *Exponentially-Weighted Least Squares* (~35 min)
**Goal:** derive the RLS recursion from the matrix inversion lemma; implement it.
**Builds on:** [APA](./Intro_AdFilt_APA.ipynb). &nbsp; **Feeds into:** Session 2 (RLS ↔ Kalman).

---

## 2. Solving ALL of History, Every Sample

💡 **Intuition.** LMS/NLMS/APA use a *window* of data per update. RLS is greedier: at time $n$ it wants the exact minimizer of **all** past errors, forgetting old data exponentially: $J_n = \sum_{k\le n} \lambda^{n-k} e_k^2$ with forget factor $\lambda \lesssim 1$. Naively that's a matrix solve per sample. The rescue is the **matrix inversion lemma**: a rank-one update to $R$ produces a rank-one update to $R^{-1}$ — so the inverse is *carried along* and each step costs $O(M^2)$, not $O(M^3)$.

**The recursion.** Carry $P_n \approx R_n^{-1}$:
$$\mathbf{k}_n = \frac{P_{n-1}\mathbf{x}_n}{\lambda + \mathbf{x}_n^T P_{n-1}\mathbf{x}_n} \qquad e_n = d_n - \mathbf{w}_{n-1}^T \mathbf{x}_n$$
$$\mathbf{w}_n = \mathbf{w}_{n-1} + \mathbf{k}_n e_n \qquad P_n = \lambda^{-1}\big(P_{n-1} - \mathbf{k}_n \mathbf{x}_n^T P_{n-1}\big)$$

Same heartbeat as ever — *new = old + gain × error* — but the gain $\mathbf{k}_n$ now carries the full curvature of history, so convergence is nearly immune to input correlation (no more $\kappa$ penalty from [Optimization S2](../Intro_Math/Optimization/Optimization.ipynb)).

In [2]:
def rls(x, d, M, lam=0.999, delta=100.0):
    w = np.zeros(M); P = delta*np.eye(M); e = np.zeros(len(x)); xb = np.zeros(M)
    for n_i in range(len(x)):
        xb = np.roll(xb, 1); xb[0] = x[n_i]
        Px = P @ xb
        k = Px / (lam + xb @ Px)
        e[n_i] = d[n_i] - w @ xb
        w = w + k * e[n_i]
        P = (P - np.outer(k, Px)) / lam
    return w, e

def nlms(x, d, M, mu=0.5, eps=1e-6):
    w = np.zeros(M); e = np.zeros(len(x)); xb = np.zeros(M)
    for n_i in range(len(x)):
        xb = np.roll(xb, 1); xb[0] = x[n_i]
        e[n_i] = d[n_i] - w @ xb
        w = w + mu/(eps + xb@xb) * e[n_i] * xb
    return w, e

w_rls, e_rls = rls(x, d, M)
w_nlms, e_nlms = nlms(x, d, M)

def curve(e): return 10*np.log10(np.convolve(e**2, np.ones(60)/60, "valid") + 1e-12)
plt.figure(figsize=(8, 3))
plt.plot(curve(e_nlms), label="NLMS (correlated input hurts)")
plt.plot(curve(e_rls), label="RLS (carries the inverse: barely notices)")
plt.legend(); plt.grid(True, alpha=0.3); plt.xlabel("sample"); plt.ylabel("MSE [dB]")
plt.title("Correlated input: RLS converges in ~2M samples")
plt.tight_layout(); plt.show()
print(f"weight error  RLS {np.linalg.norm(w_rls-w_true):.5f}   NLMS {np.linalg.norm(w_nlms-w_true):.5f}")

weight error  RLS 0.00137   NLMS 0.00430


/tmp/ipykernel_2040437/3591652342.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *RLS ↔ Kalman* (~35 min)
**Goal:** see RLS as a Kalman filter for a static state; know the cost/robustness trade-table.
**Builds on:** Session 1; [Kalman](./Intro_AdFilt_KF.ipynb).

---

## 3. The Identification

💡 **Intuition.** Stare at the RLS recursion next to the [Kalman equations](./Intro_AdFilt_KF.ipynb): they are the *same algorithm*. Model the weights as a **static hidden state** ($F = I$, $Q = 0$) observed through $d_n = \mathbf{x}_n^T \mathbf{w} + v_n$ (so $H = \mathbf{x}_n^T$ changes every step): the Kalman gain becomes $\mathbf{k}_n$, the covariance $P$ is RLS's $P$, and $\lambda < 1$ plays the role of process noise — a confession that the 'static' weights actually drift. One framework, three names: RLS (filtering), recursive least squares (statistics), Kalman with random regressors (control).

In [3]:
# Verify the identification numerically: Kalman-with-static-state ≡ RLS (λ=1)
def kalman_static(x, d, M, R_meas=1.0, P0=100.0):
    w = np.zeros(M); P = P0*np.eye(M); xb = np.zeros(M)
    for n_i in range(len(x)):
        xb = np.roll(xb, 1); xb[0] = x[n_i]
        S = xb @ P @ xb + R_meas
        k = P @ xb / S
        w = w + k * (d[n_i] - w @ xb)
        P = P - np.outer(k, xb @ P)
    return w

w_kal = kalman_static(x, d, M)
w_rls1, _ = rls(x, d, M, lam=1.0, delta=100.0)
print("max |w_Kalman − w_RLS(λ=1)| =", np.abs(w_kal - w_rls1).max())

max |w_Kalman − w_RLS(λ=1)| = 3.1580640880157773e-15


### The Family Portrait

| | LMS | NLMS | APA-$K$ | RLS | Kalman |
|---|---|---|---|---|---|
| Cost/sample | $O(M)$ | $O(M)$ | $O(K^2M)$ | $O(M^2)$ | $O(M^2)$+model |
| Colored-input speed | ✗ | ✗ | ○ | ✓ | ✓ |
| Tracks drifting systems | ✓ | ✓ | ✓ | via $\lambda$ | via $Q$ (principled) |
| Needs a state model | – | – | – | – | **yes** |
| Numerical fragility | robust | robust | mild | $P$ can lose symmetry* | same, use square-root forms |

\*Production RLS/Kalman uses QR/square-root updates — the [SOS lesson](../Intro_DSP/Filter_Design.ipynb) again: factor, don't invert.

## 4. Conclusion

RLS = exact least squares carried recursively via the matrix inversion lemma = Kalman filtering a static state. The whole adaptive-filtering ladder is one algorithm at increasing levels of self-knowledge.

---
## Where next

- [Beyond Kalman](./Beyond_Kalman.ipynb) — when the state *moves nonlinearly*.
- [Kernel Methods](../Intro_Mach_Learn/Kernel_Methods.ipynb) — KRLS: this recursion in a feature space.